In [ ]:
import json
import pennylane as qml
import pennylane.numpy as np

dev = qml.device('default.qutrit', wires = 1)

@qml.qnode(dev)
def prepare_qutrit(chi, eta):
    """
    This QNode prepares the state |phi> as defined in the statement and
    computes the measurement probabilities in the qutrit computational basis.
    
    Args: 
        - chi (float): The angle chi parametrizing the state |phi>.
        - eta (float): The angle eta parametrizing the state |eta>.
    Returns:
        - (np.array(float)): The measurement probabilities in the computational
        basis after preparing the state.
    
    """
    # Define the coefficients of the state |phi>
    c0 = np.sin(chi) * np.cos(eta)
    c1 = np.sin(chi) * np.sin(eta)
    c2 = np.cos(chi)
    
    # Create the state vector
    state_vector = np.array([c0, c1, c2], dtype=np.complex128)
    
    # Prepare the state using qml.StatePrep
    qml.StatePrep(state_vector, wires=0)

    # Return the measurement probabilities in the computational basis
    return qml.probs(wires=0)


def evaluate_sum(chi, eta_array):
    """
    This QNode computes the sum S as in the statement.
    
    Args: 
        chi (float): The angle chi parametrizing the states |phi_i>.
        eta_array (float): Contains the angles eta_i parametrizing the state |eta_i>.
    Returns:
        (np.array(float)): The sum S as defined in the statement.
        
    """
    total_sum = 0.0
    # Iterate over each eta_j in the eta_array
    for eta_j in eta_array:
        # Prepare the state |phi_j> and get its measurement probabilities
        probabilities = prepare_qutrit(chi, eta_j)
        
        # The term |<psi | phi_j>|^2, where |psi> = |2>, is the probability of measuring |2>
        # This corresponds to the third element (index 2) of the probabilities array.
        prob_psi_phi_j = probabilities[2]
        
        # Add this probability to the total sum
        total_sum += prob_psi_phi_j
        
    return total_sum


# These functions are responsible for testing the solution.


def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = evaluate_sum(*ins)
    
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output,expected_output, atol = 1e-4), "Not the correct sum!"